In [3]:
# download the helper files
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

In [1]:
# from 01-generate-truth.ipynb

# import the required packages
from ingest import load_faq_data
from evaluation_utils import llm_structured_retry, calc_price
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv
from tqdm.auto import tqdm
from pprint import pprint
import pandas as pd
import json
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc['course']=='llm-zoomcamp':
        documents_llm.append(doc)

documents = documents_llm 

In [2]:
class Questions(BaseModel):
    questions: list[str]

def generate_ground_truth(llm_client, doc, data_gen_instructions, output_type=Questions, model='gpt-4o-mini'):
    user_prompt = json.dumps(doc)

    result, usage = llm_structured_retry(
        client=llm_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=output_type,
        model=model
    )

    records = []
    for q in result.questions:
        records.append({
            'question':q,
            'document':doc['id']
        })
    
    return records, usage

In [3]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

llm_client = OpenAI(api_key=api_key)

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(llm_client=llm_client, doc=doc, data_gen_instructions=data_gen_instructions)
    ground_truth.extend(records)
    usages.append(usage)

# the for loop runs one LLM call after another - running it for all documents this way would take too long

  0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
# running the calls one after another wastes most of the time waiting on the network - each request just sits there until OpenAI responds, 
# so we can fire several at once and wait on them together - we process the documents in parallel and track progress while the requests run
# Note: don't open too many connections at once, or you'll hit the provider's rate limits - five or six workers is a safe default here

# replace the loop with parallel jobs
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress, calc_total_price

def process_doc(doc):
    return generate_ground_truth(
        llm_client=llm_client,
        doc=doc,
        data_gen_instructions=data_gen_instructions
    )

ground_truth = []
usages = []

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, process_doc)

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

print(f'Ground Truths Generated: {len(ground_truth)}')
print(f'Total Cost: {calc_total_price(usages=usages)}')

  0%|          | 0/113 [00:00<?, ?it/s]

Ground Truths Generated: 565
Total Cost: 0.07893899999999998


In [8]:
# save the ground truth to a csv file
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)